<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/ml-fundamentals/gradient-boosting/gradient-boosting.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("gradient-boosting", ...)`


# Gradient Boosting Regressor

**Difficulty**: 🔴 Hard

---

### Problem Statement

Gradient-boosted decision trees (XGBoost, LightGBM, CatBoost) are still the strongest models for
tabular data: ranking and ads features, fraud, forecasting, and many Kaggle wins. In ML-breadth
interviews you're expected to explain where the *gradient* in gradient boosting comes from and to
implement it.

Boosting builds an additive model $F_M(x) = F_0 + \nu\sum_{m=1}^{M} h_m(x)$ one small tree at a
time. Each new tree $h_m$ is fit to the **negative gradient** of the loss with respect to the
current predictions. For the squared loss $L = \tfrac12\,(y - F)^2$ that negative gradient is the
**residual** $y - F(x)$. The learning rate $\nu$ ("shrinkage") scales every step.

You will write the boosting loop **and** the weak learner: a CART regression tree whose splits
minimize squared error and whose leaves predict the mean residual.

---

### Requirements

1. **`GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2)`**
2. **`fit(X, y)`**: `X` is `(N, D)` float, `y` is `(N,)` float. Returns `self`. Sets
   **`init_`** (a float) to $F_0 = \operatorname{mean}(y)$.
3. **Boosting loop** for $m = 1, \dots,$ `n_estimators`:
   - residuals $r_i = y_i - F_{m-1}(x_i)$;
   - fit a regression tree $h_m$ to $(X, r)$;
   - $F_m = F_{m-1} + \nu\,h_m$.
4. **Regression tree** (your own; a helper class is fine):
   - binary splits `x[f] <= t`, with candidate thresholds at the **midpoints between consecutive
     distinct sorted values** of the feature among the node's samples;
   - choose the split minimizing $\mathrm{SSE}_L + \mathrm{SSE}_R$ (variance reduction). Split only
     if that is **strictly lower** than the node's own SSE;
   - a leaf predicts the **mean** target (residual) of its samples;
   - `max_depth` (root = depth 0, `None` = unlimited) and `min_samples_split` (a node with fewer
     samples is a leaf), as in CART.
5. **`predict(X)`** → `(M,)`: $F_0 + \nu \sum_m h_m(X)$.
6. **`staged_predict(X)`**: a list or generator of `n_estimators` tensors of shape `(M,)`. Entry
   `i` is the prediction using the first `i + 1` trees, *not* including the bare $F_0$. Each entry
   must be its own tensor, so `list(staged_predict(X))` is correct.

---

### Constraints

- ✅ PyTorch tensor ops. Loops over boosting stages and tree nodes are fine.
- ✅ Vectorized split search: one sort + cumulative sums per node, not a loop over thresholds.
- ❌ No scikit-learn, XGBoost or LightGBM.

---

<details>
  <summary>💡 Hint</summary>

  - Which constant $c$ minimizes $\sum_i (y_i - c)^2$? That's your $F_0$.
  - $\mathrm{SSE} = \sum r^2 - S^2/n$, where $S = \sum r$. Sort a feature once: for the cut after
    sorted position $i$, $S_L$ is a prefix sum of the sorted residuals and $S_R = S - S_L$.
  - Minimizing $\mathrm{SSE}_L + \mathrm{SSE}_R$ is the same as maximizing $S_L^2/n_L + S_R^2/n_R$.
    Mask cuts between equal feature values.
  - Why is the mean the right leaf value for squared loss? What would it be for absolute loss?
  - With squared loss and $0 < \nu \le 1$, should the training MSE ever go up from one stage to the next?

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# A noisy sine wave: easy to see, impossible for a single shallow tree.
torch.manual_seed(0)
X = torch.rand(300, 1) * 2 * math.pi                 # (300, 1)
y = torch.sin(X[:, 0]) + 0.1 * torch.randn(300)      # (300,), noise variance 0.01
X_train, y_train, X_test, y_test = X[:200], y[:200], X[200:], y[200:]
print("X_train", tuple(X_train.shape), "y_train", tuple(y_train.shape))

In [ ]:
class RegressionTree:
    """(Suggested helper) CART regression tree: SSE-minimizing splits, mean-valued leaves."""

    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, r):
        """X: (N, D) features, r: (N,) targets. Returns self."""
        # TODO
        raise NotImplementedError

    def predict(self, X):
        """(M, D) -> (M,)."""
        # TODO
        raise NotImplementedError


class GradientBoostingRegressor:
    """Gradient-boosted regression trees for squared loss."""

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        """
        Args:
            X: (N, D) features.
            y: (N,) targets.
        Returns:
            self (with init_ = mean(y) and the fitted trees)
        """
        # TODO: F_0 = mean(y); for each stage fit a tree to the residuals y - F and take a
        #       shrunken step F <- F + learning_rate * tree(X)
        raise NotImplementedError

    def staged_predict(self, X):
        """Yield (or return a list of) the (M,) prediction after each stage."""
        # TODO
        raise NotImplementedError

    def predict(self, X):
        """(M, D) -> (M,)."""
        # TODO
        raise NotImplementedError

In [ ]:
# Test 1: one stage of depth 1 with learning_rate=1 == the best regression stump (brute force)
def brute_force_stump(X, y):
    y = y.double()
    best = (float("inf"), None, None)
    for f in range(X.shape[1]):
        vals = torch.unique(X[:, f])
        for t in ((vals[:-1] + vals[1:]) / 2).tolist():
            m = X[:, f] <= t
            sse = (((y[m] - y[m].mean()) ** 2).sum() + ((y[~m] - y[~m].mean()) ** 2).sum()).item()
            if sse < best[0]:
                best = (sse, f, t)
    return best

sse, f, t = brute_force_stump(X_train, y_train)
stump = GradientBoostingRegressor(n_estimators=1, learning_rate=1.0, max_depth=1).fit(X_train, y_train)
left = X_train[:, f] <= t
want = torch.where(left, y_train[left].mean(), y_train[~left].mean())
assert abs(stump.init_ - y_train.mean().item()) < 1e-6, "init_ must be mean(y)"
assert torch.allclose(stump.predict(X_train), want, atol=1e-5), "not the best SSE stump"
print(f"best stump: x[{f}] <= {t:.4f}, SSE {sse:.3f}")
print("Test 1 passed")

In [ ]:
# Test 2: training MSE never goes up across stages; the full model fits the sine
gbr = GradientBoostingRegressor().fit(X_train, y_train)
staged = list(gbr.staged_predict(X_train))
assert len(staged) == gbr.n_estimators, f"expected {gbr.n_estimators} stages, got {len(staged)}"
assert torch.allclose(staged[-1], gbr.predict(X_train), atol=1e-6)
mse = [((p - y_train) ** 2).mean().item() for p in staged]
assert all(b <= a * (1 + 1e-5) + 1e-7 for a, b in zip(mse[:-1], mse[1:])), "training MSE went up"
test_mse = ((gbr.predict(X_test) - y_test) ** 2).mean().item()
print("train MSE after 1, 10, 50, 100 stages:", [round(mse[i], 4) for i in (0, 9, 49, 99)])
print(f"test MSE {test_mse:.4f} (noise variance 0.01)")
assert mse[-1] < 0.015 and test_mse < 0.03
print("Test 2 passed")

In [ ]:
# Test 3: the learning rate scales the first stage's update exactly
a = GradientBoostingRegressor(n_estimators=1, learning_rate=1.0).fit(X_train, y_train)
b = GradientBoostingRegressor(n_estimators=1, learning_rate=0.1).fit(X_train, y_train)
base = y_train.mean()
assert torch.allclose(b.predict(X_test) - base, 0.1 * (a.predict(X_test) - base), atol=1e-6)
zero = GradientBoostingRegressor(n_estimators=5, learning_rate=0.0).fit(X_train, y_train)
assert torch.allclose(zero.predict(X_test), base.expand(len(X_test)))
print("Test 3 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("gradient-boosting", GradientBoostingRegressor=GradientBoostingRegressor)